# User Retention and Churn Analysis
## Notebook 1: Data Exploration

**Author:** Vijayalakshmi Veeraiyan  
**Dataset:** IBM Telecom Customer Churn (Kaggle)  
**Goal:** Understand the dataset structure, identify missing values, and explore key patterns that might predict customer churn.

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
plt.style.use('seaborn-v0_8')

print('Libraries loaded successfully!')

## 2. Load the Dataset

Upload `telecom_customer_churn.csv` to your Google Colab session before running this cell.

In [ ]:
# Load the dataset
file_path = '/content/telecom_customer_churn.csv'
df = pd.read_csv(file_path)

print(f'Dataset loaded successfully!')
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')

## 3. Basic Dataset Overview

In [ ]:
# First 5 rows
print('First 5 rows of the dataset:')
df.head()

In [ ]:
# Data types
print('Column data types:')
print(df.dtypes)

In [ ]:
# Basic statistics for numerical columns
print('Summary statistics:')
df.describe()

## 4. Missing Values Analysis

In [ ]:
# Count missing values per column
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})

# Show only columns with missing values
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
print('Columns with missing values:')
print(missing_df)

In [ ]:
# Visualise missing values
plt.figure(figsize=(10, 6))
missing_df['Missing %'].plot(kind='bar', color='steelblue')
plt.title('Missing Values by Column (%)', fontsize=14)
plt.xlabel('Column')
plt.ylabel('Missing %')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('missing_values.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as missing_values.png')

## 5. Target Variable Analysis — Churn Distribution

In [ ]:
# Churn distribution
churn_counts = df['Customer Status'].value_counts()
churn_pct = df['Customer Status'].value_counts(normalize=True) * 100

print('Customer Status Distribution:')
for status in churn_counts.index:
    print(f'  {status}: {churn_counts[status]} customers ({churn_pct[status]:.1f}%)')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
axes[0].bar(churn_counts.index, churn_counts.values, color=['steelblue', 'salmon', 'green'])
axes[0].set_title('Customer Status Count', fontsize=13)
axes[0].set_ylabel('Number of Customers')

# Pie chart
axes[1].pie(churn_counts.values, labels=churn_counts.index, autopct='%1.1f%%',
            colors=['steelblue', 'salmon', 'green'])
axes[1].set_title('Customer Status Distribution', fontsize=13)

plt.tight_layout()
plt.savefig('churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nNote: Class imbalance exists — churned customers are minority class')

## 6. Key Feature Exploration

In [ ]:
# Create binary churn flag for analysis
df['Churned'] = (df['Customer Status'] == 'Churned').astype(int)

# Churn rate by Contract type
contract_churn = df.groupby('Contract')['Churned'].mean() * 100
print('Churn Rate by Contract Type:')
print(contract_churn.round(1))

plt.figure(figsize=(8, 5))
contract_churn.plot(kind='bar', color='steelblue')
plt.title('Churn Rate by Contract Type (%)', fontsize=13)
plt.ylabel('Churn Rate (%)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('churn_by_contract.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Churn rate by Number of Referrals
referral_churn = df.groupby('Number of Referrals')['Churned'].mean() * 100

plt.figure(figsize=(10, 5))
referral_churn.plot(kind='bar', color='steelblue')
plt.title('Churn Rate by Number of Referrals (%)', fontsize=13)
plt.ylabel('Churn Rate (%)')
plt.xlabel('Number of Referrals')
plt.tight_layout()
plt.savefig('churn_by_referrals.png', dpi=150, bbox_inches='tight')
plt.show()
print('Observation: More referrals = lower churn rate (strong loyalty signal)')

In [ ]:
# Distribution of Monthly Charge for churned vs non-churned customers
churned = df[df['Churned'] == 1]['Monthly Charge']
stayed = df[df['Churned'] == 0]['Monthly Charge']

plt.figure(figsize=(10, 5))
plt.hist(stayed, bins=40, alpha=0.6, label='Stayed', color='steelblue')
plt.hist(churned, bins=40, alpha=0.6, label='Churned', color='salmon')
plt.title('Monthly Charge Distribution: Churned vs Stayed', fontsize=13)
plt.xlabel('Monthly Charge ($)')
plt.ylabel('Number of Customers')
plt.legend()
plt.tight_layout()
plt.savefig('monthly_charge_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Average monthly charge - Churned: ${churned.mean():.2f}')
print(f'Average monthly charge - Stayed: ${stayed.mean():.2f}')

In [ ]:
# Churn rate by Tenure group
df['Tenure_Group'] = pd.cut(df['Tenure in Months'],
                             bins=[0, 12, 24, 36, 48, 72],
                             labels=['0-12m', '13-24m', '25-36m', '37-48m', '49-72m'])

tenure_churn = df.groupby('Tenure_Group', observed=True)['Churned'].mean() * 100

plt.figure(figsize=(8, 5))
tenure_churn.plot(kind='bar', color='steelblue')
plt.title('Churn Rate by Tenure Group (%)', fontsize=13)
plt.ylabel('Churn Rate (%)')
plt.xlabel('Tenure Group')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('churn_by_tenure.png', dpi=150, bbox_inches='tight')
plt.show()
print('Observation: Newer customers churn more — early retention is critical')

In [ ]:
# Correlation heatmap for numerical features
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
# Remove latitude, longitude, zip code
numerical_cols = [c for c in numerical_cols if c not in ['Latitude', 'Longitude', 'Zip Code']]

plt.figure(figsize=(14, 10))
corr_matrix = df[numerical_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, annot_kws={'size': 8})
plt.title('Correlation Heatmap — Numerical Features', fontsize=13)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Key Observations Summary

From this exploratory analysis, the following patterns are clear:

1. **Class imbalance exists** — ~26.5% of customers churned, ~73.5% stayed. This needs to be handled in modeling.
2. **Month-to-Month contracts have the highest churn rate** — customers on flexible contracts leave more easily.
3. **More referrals = lower churn** — customers who refer others are significantly more loyal.
4. **Churned customers pay more monthly** — higher charges correlate with higher churn risk.
5. **New customers churn more** — early tenure period is the most critical for retention.

These insights will guide our feature engineering in Notebook 2.

---
**Next:** Open `2_preprocessing.ipynb` to clean and prepare the data for modeling.